In [1]:
print("test")

test


In [2]:
# ---- 1. Setup: pinned training deps ----
!pip install -q transformers==5.16.1 peft==0.20.0 trl==1.12.0 accelerate==1.14.0 datasets==5.0.1 bitsandbytes jiwer
# torch: use Colab's preinstalled CUDA build (do not reinstall)
import torch, transformers, peft, trl
print(torch.__version__, transformers.__version__, peft.__version__, trl.__version__)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 24.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.9 MB/s eta 0:00:00
2.11.0+cu128 5.16.1 0.20.0 1.12.0
Tesla T4


In [3]:
# first we need to log into google drive and hugging face
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# then we need to log into hugging face
from huggingface_hub import login
login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [ ]:
from pathlib import Path

BASE_MODEL = 'Qwen/Qwen3.5-2B'
DATA_DIR = Path('/content/drive/MyDrive/Model Training Pipeline Final')  # upload the folder to Drive
CKPT_DIR = Path('/content/drive/MyDrive/rambler_checkpoints_2b')
EVAL_OUT = CKPT_DIR / 'eval_report.json'

MAX_SEQ_LEN = 256          # utterances are short (plan §5)
MICRO_BS = 8               # Reduced from 16 to avoid OOM on T4
GRAD_ACCUM = 4             # Increased from 2 to keep effective batch size = 32
LR = 2e-4
NUM_EPOCHS = 3
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
SEED = 42
BENCH_STEPS = 30
EVAL_N = 300               # validation rows to generate on (None = all 1500)
SESSION_BUDGET_MIN = 150   # stay well under Colab's ~3h limit (plan §5.1)
RELOAD_ADAPTER = False     # True after a session restart to skip retraining

# This exact string becomes part of the shipped app prompt. Keep it stable.
SYSTEM_PROMPT = (
    "Clean up this voice dictation: remove all filler words, stutters, false starts, "
    "and introductory conversational rambling (e.g., 'I was thinking', 'well basically'); "
    "resolve self-corrections; fix punctuation and capitalization. Keep every core fact, "
    "name, date, time, and number. Output only the cleaned text."
)

CKPT_DIR.mkdir(parents=True, exist_ok=True)
print('config ok | data dir exists:', DATA_DIR.exists())

config ok | data dir exists: True


In [ ]:
import torch

BACKEND = None
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,            # auto (fp16 on T4)
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        # Gated DeltaNet layers have no q/k/v/o projections -> they stay
        # frozen; attention+MLP adaptation is enough for this task size.
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )
    BACKEND = 'unsloth'
    print('backend: Unsloth')
except Exception as e:
    print('Unsloth failed -> HF peft fallback (plan §5.2). Reason:')
    print(repr(e)[:500])
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map={'': 0}
    )
    model.config.use_cache = False

    # Force cast all non-quantized parameters (norms/embeddings) to float32
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    
    for name, module in model.named_modules():
        if "norm" in name.lower():
            module.to(torch.float32)
        if "lm_head" in name.lower() or "embed_tokens" in name.lower():
            if hasattr(module, "weight") and module.weight.dtype == torch.bfloat16:
                module.to(torch.float32)

    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
    ))
    
    # Manually ensure all trainable adapters are float32 for the GradScaler
    for param in model.parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)
            
    BACKEND = 'peft'
    print('backend: HF peft')

if RELOAD_ADAPTER:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(CKPT_DIR / 'adapter_final'))
    print('reload mode: attached adapter_final from Drive (skip training)')

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.print_trainable_parameters()

Unsloth failed -> HF peft fallback (plan §5.2). Reason:
ModuleNotFoundError("No module named 'unsloth'")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

backend: HF peft
trainable params: 10,911,744 || all params: 1,892,736,832 || trainable%: 0.5765


In [7]:
# ---- 6. Dataset: JSONL -> chat-template text field ----
import json
import random
from datasets import Dataset

def resolve(name):
    for cand in (DATA_DIR / name, Path(name)):
        if cand.exists():
            return cand
    raise FileNotFoundError(f'{name} not found in Drive ({DATA_DIR}) or cwd')

def render_chat(user_text, assistant_text):
    return tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': user_text},
         {'role': 'assistant', 'content': assistant_text}],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )

DATA = {}
for split, fname in [('train', 'train.jsonl'), ('validation', 'validation.jsonl')]:
    path = resolve(fname)
    DATA[split] = [json.loads(l) for l in path.read_text(encoding='utf-8').splitlines() if l.strip()]
    print(split, len(DATA[split]), 'rows from', path)

train_ds = Dataset.from_list(
    [{'text': render_chat(r['input'], r['output'])} for r in DATA['train']]
)
print('\n--- rendered sample ---')
print(train_ds[0]['text'][:500])

train 12750 rows from /content/drive/MyDrive/Model Training Pipeline Final/train.jsonl
validation 1500 rows from /content/drive/MyDrive/Model Training Pipeline Final/validation.jsonl

--- rendered sample ---
<|im_start|>system
Clean up this voice dictation: remove filler words, stutters, and self-corrections; fix punctuation and capitalization. Keep every fact, name, date, time, and number exactly as dictated. Output only the cleaned text.<|im_end|>
<|im_start|>user
My reason for that was I don't like the, what's the right word. The varied inappropriate influences that you find so much in the public schools.<|im_end|>
<|im_start|>assistant
<think>

</think>

My reason for that was I don't like the, 


In [ ]:
# # ---- Benchmark-first
# # These BENCH_STEPS also act as a warmup; the real run continues from here.
# import time
# from trl import SFTTrainer, SFTConfig

# bench_args = SFTConfig(
#     output_dir=str(CKPT_DIR / 'bench'),
#     per_device_train_batch_size=MICRO_BS,
#     gradient_accumulation_steps=GRAD_ACCUM,
#     max_steps=BENCH_STEPS,
#     learning_rate=LR,
#     logging_steps=5,
#     save_strategy='no',
#     fp16=False,                 # <--- Set to False (bypasses GradScaler)
#     bf16=False,
#     max_length=MAX_SEQ_LEN,
#     packing=False,
#     dataset_text_field='text',
#     seed=SEED,
#     report_to='none',
# )
# bench = SFTTrainer(model=model, args=bench_args, train_dataset=train_ds,
#                    processing_class=tokenizer)
# t0 = time.time()
# bench.train()
# dt = time.time() - t0

# steps_per_sec = BENCH_STEPS / dt
# steps_per_epoch = -(-len(train_ds) // (MICRO_BS * GRAD_ACCUM))  # ceil
# epoch_min = steps_per_epoch / steps_per_sec / 60
# total_min = epoch_min * NUM_EPOCHS + 15  # +15 min headroom for eval/export
# print(f'{steps_per_sec:.2f} steps/s | {steps_per_epoch} steps/epoch')
# print(f'epoch ~= {epoch_min:.1f} min | {NUM_EPOCHS} epochs + eval ~= {total_min:.1f} min')
# print('GO: proceed to training' if total_min < SESSION_BUDGET_MIN else
#       'NO-GO: reduce NUM_EPOCHS/EVAL_N or switch GPU (plan §5.1)')

In [ ]:
# ---- 8. Resume-awareness (plan §5.1): newest Drive checkpoint wins ----
import re

RESUME_FROM = None
checkpoints = sorted(
    CKPT_DIR.glob('checkpoint-*'),
    key=lambda p: int(re.findall(r'\d+', p.name)[0]),
) if CKPT_DIR.exists() else []
if checkpoints and not RELOAD_ADAPTER:
    RESUME_FROM = str(checkpoints[-1])
    print('resuming from', RESUME_FROM)
else:
    print('fresh run (no checkpoints found)' if not checkpoints else 'reload mode: skipping resume')

resuming from /content/drive/MyDrive/rambler_checkpoints_2b/checkpoint-1100


In [10]:
# ---- 9. Train: step-based checkpoints (every 100 steps) straight to Drive ----
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=MICRO_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=40,            # Replaces warmup_ratio=0.03
    weight_decay=0.01,
    logging_steps=20,
    save_strategy='steps',
    save_steps=100,             # checkpoint every 100 steps so progress is never lost
    save_total_limit=4,
    fp16=False,                 # Remember to keep fp16/bf16 False here too
    bf16=False,
    max_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field='text',
    seed=SEED,
    report_to='none',
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     processing_class=tokenizer)

if BACKEND == 'unsloth':
    try:
        from unsloth.chat_templates import train_on_responses_only
        trainer = train_on_responses_only(
            trainer,
            instruction_part='<|im_start|>user\n',
            response_part='<|im_start|>assistant\n',
        )
        print('loss masked to assistant tokens')
    except Exception as e:
        print('WARN: response masking unavailable -> full-sequence loss.', repr(e)[:200])
        print('Check the rendered sample above for the real ChatML markers.')

trainer.train(resume_from_checkpoint=RESUME_FROM)

Adding EOS to train dataset:   0%|          | 0/12750 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12750 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/12750 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12750 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/12750 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss
1120,0.373217
1140,0.379179
1160,0.375740
1180,0.382211


TrainOutput(global_step=1197, training_loss=0.03063742120562739, metrics={'train_runtime': 1137.9579, 'train_samples_per_second': 33.613, 'train_steps_per_second': 1.052, 'total_flos': 4.346261341310131e+16, 'train_loss': 0.03063742120562739, 'entropy': 0.4067603123910499, 'num_tokens': 342762.0, 'mean_token_accuracy': 0.9129765756202467, 'epoch': 3.0})

In [11]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## 11. Merge adapter into base model

**Fixed:** removed the old `config.json` patch that zeroed `num_nextn_predict_layers` / `nextn_predict_layers` — those keys don't exist on this model (the real key is `mtp_num_hidden_layers`), so it was silent dead code. Added diagnostics so we can see exactly what the MTP module looks like before deciding how to handle it in GGUF conversion.

In [1]:
# ---- 11. Merge LoRA adapter into base model (cleaned up) ----
import gc
import json
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import peft.import_utils

peft.import_utils.is_torchao_available = lambda: False

BASE_MODEL = globals().get('BASE_MODEL', 'Qwen/Qwen3.5-2B')
CKPT_DIR = Path('/content/drive/MyDrive/rambler_checkpoints_2b')
final_dir = CKPT_DIR / 'adapter_final'
merged_dir = CKPT_DIR / 'merged_fp16'
(CKPT_DIR / 'gguf').mkdir(parents=True, exist_ok=True)

# 1. Fallback for tokenizer if session was restarted
if 'tokenizer' not in globals() or tokenizer is None:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# 2. Persist adapter if active in memory
if 'model' in globals() and model is not None:
    model.save_pretrained(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print(f'LoRA adapter saved -> {final_dir}')
    del model
    gc.collect()
    torch.cuda.empty_cache()

# 3. Reload base model in FP16 on CPU and merge
print('Reloading base model in float16 to merge...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='cpu',
    low_cpu_mem_usage=True,
)

lora_model = PeftModel.from_pretrained(base_model, str(final_dir))
merged_model = lora_model.merge_and_unload()

merged_model.config.mtp_num_hidden_layers = 0

# ---- FIX: diagnostics instead of blindly patching config ----
print('config.num_hidden_layers:', merged_model.config.num_hidden_layers)
mtp_fields = {k: v for k, v in merged_model.config.to_dict().items()
              if 'nextn' in k.lower() or 'mtp' in k.lower()}
print('MTP-related config fields:', mtp_fields)
print('physical layers in model.layers:', len(merged_model.model.layers))

print('\n--- Locating MTP module(s) ---')
mtp_modules = []
for name, module in merged_model.named_modules():
    if 'mtp' in name.lower():
        mtp_modules.append((name, type(module).__name__))
        print(name, '->', type(module).__name__)
if not mtp_modules:
    print('No dedicated MTP module found by name -- it may be folded into another '
          'attribute. Inspect `merged_model` manually if mtp_fields above shows '
          'mtp_num_hidden_layers > 0.')

# ---- Nothing is deleted/patched automatically here. ----
# Once you've confirmed the module path from the printout above, decide between:
#   (a) leave MTP in place and rely on an up-to-date llama.cpp build that natively
#       supports qwen35 NextN/MTP tensors, or
#   (b) explicitly remove the module and set the REAL key
#       (merged_model.config.mtp_num_hidden_layers = 0) before saving, e.g.:
#       del merged_model.<mtp_attribute_path>
#       merged_model.config.mtp_num_hidden_layers = 0
# Do this only after confirming the attribute path -- don't guess.

print(f'\nSaving merged FP16 model to {merged_dir}...')
merged_model.save_pretrained(str(merged_dir))
tokenizer.save_pretrained(str(merged_dir))
print('Merge complete.')


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Reloading base model in float16 to merge...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

config.num_hidden_layers: 24
MTP-related config fields: {'mtp_num_hidden_layers': 0, 'mtp_use_dedicated_embeddings': False}
physical layers in model.layers: 24

--- Locating MTP module(s) ---
No dedicated MTP module found by name -- it may be folded into another attribute. Inspect `merged_model` manually if mtp_fields above shows mtp_num_hidden_layers > 0.

Saving merged FP16 model to /content/drive/MyDrive/rambler_checkpoints_2b/merged_fp16...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merge complete.


## 12. Build llama.cpp (clean, no hand-patching)

**Fixed:** removed the `convert_hf_to_gguf.py` regex patch (it was rewriting a line that apparently doesn't control what we thought — the emitted `nextn_predict_layers` value in cell 14's old read-back was unaffected by it). Also removed the raw GGUF binary patcher entirely — patching header bytes without removing the underlying tensors just created a file whose metadata contradicted its own contents, which is exactly what broke `llama-cli` in cell 17. We now run the **unmodified** converter and see what it produces on its own.

In [2]:
%%bash
# Install Python conversion dependencies
pip install -q gguf sentencepiece protobuf

# Fresh, unpatched clone every time -- do not hand-edit convert_hf_to_gguf.py
rm -rf llama.cpp
git clone --depth 1 https://github.com/ggml-org/llama.cpp.git

cd llama.cpp
cmake -B build -DGGML_CUDA=OFF
cmake --build build --config Release -j$(nproc) --target llama-quantize llama-cli
echo "Build complete: llama-quantize and llama-cli ready in ./llama.cpp/build/bin/"


-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- llama.cpp version: 0.4.0-dev
-- Found Git: /usr/bin/git (found version "2.43.0")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Found OpenMP_C: -fopenmp (found ve

Cloning into 'llama.cpp'...
Updating files: 100% (3583/3583), done.
CMAKE_BUILD_TYPE=Release


## 13. Convert merged model to FP16 GGUF without MTP

The `--no-mtp` option excludes the optional NextN/MTP draft block. This is required for this merged checkpoint because it has the 24-layer trunk but no `blk.24.*` MTP tensors.

In [3]:
%%bash
MERGED_DIR="/content/drive/MyDrive/rambler_checkpoints_2b/merged_fp16"
GGUF_DIR="/content/drive/MyDrive/rambler_checkpoints_2b/gguf"
FP16_GGUF="$GGUF_DIR/rambler-2b-f16-no-mtp.gguf"

# Convert weights to FP16 GGUF format without the optional MTP/NextN draft block.
# This prevents a phantom blk.24 declaration when the merged checkpoint has no MTP tensors.
python llama.cpp/convert_hf_to_gguf.py "$MERGED_DIR" \
  --outfile "$FP16_GGUF" \
  --outtype f16 \
  --no-mtp

echo "FP16 GGUF created at: $FP16_GGUF"

FP16 GGUF created at: /content/drive/MyDrive/rambler_checkpoints_2b/gguf/rambler-2b-f16-no-mtp.gguf


INFO:hf-to-gguf:Loading model: merged_fp16
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: Qwen3_5ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                    torch.float16 --> F16, shape = {2048, 248320}
INFO:hf-to-gguf:blk.0.attn_norm.weight,               torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ssm_a,                          torch.float16 --> F32, shape = {16}
INFO:hf-to-gguf:blk.0.ssm_conv1d.weight,              torch.float16 --> F32, shape = {4, 6144}
INFO:hf-to-gguf:blk.0.ssm_dt.bias,                    torch.float16 --> F32, shape = {16}
INFO:hf-to-gguf:blk.0.ssm_alpha.weight,               torch.float16 --> F16, shape = {2048, 16}
INFO:hf-to-gguf:blk.0.ssm_beta.weight,                torch.float16 --> F16, shape = {2048, 16}
INFO:hf-to-gguf:blk.0.att

## 14. Quantize the no-MTP GGUF to Q4_K_M

In [4]:
%%bash
GGUF_DIR="/content/drive/MyDrive/rambler_checkpoints_2b/gguf"
FP16_GGUF="$GGUF_DIR/rambler-2b-f16-no-mtp.gguf"
Q4_GGUF="$GGUF_DIR/rambler-2b-q4_k_m-no-mtp.gguf"

# Quantize the no-MTP FP16 GGUF to Q4_K_M.
./llama.cpp/build/bin/llama-quantize "$FP16_GGUF" "$Q4_GGUF" q4_k_m

echo "Quantization finished! Output saved to: $Q4_GGUF"
ls -lh "$GGUF_DIR"


llama_quantize: quantize time = 201705.26 ms
llama_quantize:    total time = 201705.26 ms
Quantization finished! Output saved to: /content/drive/MyDrive/rambler_checkpoints_2b/gguf/rambler-2b-q4_k_m-no-mtp.gguf
total 4.8G
-rw------- 1 root root 3.6G Sep 14 10:21 rambler-2b-f16-no-mtp.gguf
-rw------- 1 root root 1.2G Sep 14 10:25 rambler-2b-q4_k_m-no-mtp.gguf


version: 0.4.0-dev (build 1, commit 2f53959)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/rambler_checkpoints_2b/gguf/rambler-2b-f16-no-mtp.gguf' to '/content/drive/MyDrive/rambler_checkpoints_2b/gguf/rambler-2b-q4_k_m-no-mtp.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 33 key-value pairs and 320 tensors from /content/drive/MyDrive/rambler_checkpoints_2b/gguf/rambler-2b-f16-no-mtp.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen35
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Fp16
llama_model_loader: - kv   3:                         general.size_label str              = 1.9B
llama_model_loader: - kv   4: 

## 15. Test the no-MTP GGUF model

Run this only after the conversion and quantization cells complete successfully. If the FP16 file loads but the Q4_K_M file fails, the issue is in quantization or the runtime build rather than LoRA training.

In [5]:
!./llama.cpp/build/bin/llama-cli --help

----- common params -----

-h,    --help, --usage                  print usage and exit
--version                               show version and build info
-cl,   --cache-list                     show list of models in cache
--completion-bash                       print source-able bash completion script for llama.cpp
-t,    --threads N                      number of CPU threads to use during generation (default: -1)
                                        (env: LLAMA_ARG_THREADS)
-tb,   --threads-batch N                number of threads to use during batch and prompt processing (default:
                                        same as --threads)
-C,    --cpu-mask M                     CPU affinity mask: arbitrarily long hex. Complements cpu-range
                                        (default: "")
-Cr,   --cpu-range lo-hi                range of CPUs for affinity. Complements --cpu-mask
--cpu-strict <0|1>                      use strict CPU placement (default: 0)
--prio N           

In [14]:
!./llama.cpp/build/bin/llama-cli \
  -m "/content/drive/MyDrive/rambler_checkpoints_2b/gguf/rambler-2b-q4_k_m-no-mtp.gguf" \
  -p "<|im_start|>system\nClean up this voice dictation: remove all filler words, stutters, false starts, and introductory conversational rambling (e.g., 'I was thinking', 'well basically'); resolve self-corrections; fix punctuation and capitalization. Keep every core fact, name, date, time, and number. Output only the cleaned text.<|im_end|>\n<|im_start|>user\nUm, so basically, can you tell David that the the budget sync is not 2 PM, 4:15 PM on Tuesday afternoon. And, uh, remind Marcus to bring the, you know, pro-projector from the library, no wait, the conference room on the second floor. Also, we need to order more printer paper, coffee filters, and USB cables before Friday morning, but, I mean, don't change the shipping address on file.<|im_end|>\n<|im_start|>assistant\n" \
  -n 256 --temp 0 --single-turn --no-display-prompt --reasoning off --reasoning-budget 0



Loading model... 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b1-2f53959
model      : /content/drive/MyDrive/rambler_checkpoints_2b/gguf/rambler-2b-q4_k_m-no-mtp.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
  /read <file>        add a text file
  /glob <pattern>     add text files using globbing pattern



> <|im_start|>system
Clean up this voice dictation: remove all filler words, stutters, false starts, and introductory conversational rambling (e.g., 'I was thinking', 'well basically'); resolve self-corrections; fix punctuation and capitalization. Keep every core fact, name, date, time, and number. Output only the 

## 16. EPS eval on validation.jsonl

**Fixed:** smaller batch size, `torch.cuda.empty_cache()` between batches, and a try/except that prints the real exception instead of letting the notebook die silently (which is why we never actually saw what failed here before).

In [7]:
# ---- 16. EPS eval on validation.jsonl (hardened) ----
import re
import json
import time
import random
import collections
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import jiwer

CKPT_DIR = Path('/content/drive/MyDrive/rambler_checkpoints')
DATA_DIR = Path('/content/drive/MyDrive/Model Training Pipeline Final')
EVAL_OUT = CKPT_DIR / 'eval_report.json'
merged_dir = CKPT_DIR / 'merged_fp16'
EVAL_N = 300
SEED = 42

SYSTEM_PROMPT = (
    'Clean up this voice dictation: remove filler words, stutters, and '
    'self-corrections; fix punctuation and capitalization. Keep every fact, '
    'name, date, time, and number exactly as dictated. Output only the '
    'cleaned text. Do not reason aloud. Do not output a think block.'
)

print('Loading merged model for evaluation...')
tokenizer = AutoTokenizer.from_pretrained(str(merged_dir))
model = AutoModelForCausalLM.from_pretrained(
    str(merged_dir),
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

WORD = re.compile(r"[a-z0-9']+")

def ned(a, b):
    a, b = a.strip(), b.strip()
    if not b:
        return 0.0 if not a else 1.0
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1] / len(b)

def eps_declared(row, pred):
    ents = [v for vals in (row.get('entities') or {}).values() for v in (vals or []) if v]
    if not ents:
        return None
    out_words = set(WORD.findall(pred.lower()))
    missing = [e for e in ents if not all(w in out_words for w in WORD.findall(e.lower()))]
    return (len(ents) - len(missing)) / len(ents), missing

val_path = DATA_DIR / 'validation.jsonl'
val_rows = [json.loads(l) for l in val_path.read_text(encoding='utf-8').splitlines() if l.strip()]

rng = random.Random(SEED)
if EVAL_N and EVAL_N < len(val_rows):
    val_rows = rng.sample(val_rows, EVAL_N)
print(f'Evaluating {len(val_rows)} validation rows...')

prompts = [
    tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': '/no_think\n' + r['input']}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    ) for r in val_rows
]

old_side = tokenizer.padding_side
tokenizer.padding_side = 'left'
preds, t0 = [], time.time()
B = 4  # Reduced from 8 to prevent CUDA OOM during unquantized FP16 generation
for i in range(0, len(prompts), B):
    try:
        enc = tokenizer(prompts[i:i + B], return_tensors='pt', padding=True,
                        add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc, max_new_tokens=96, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        for j in range(enc['input_ids'].shape[0]):
            gen = out[j][enc['input_ids'].shape[1]:]
            preds.append(tokenizer.decode(gen, skip_special_tokens=True).strip())
        print(f'  {min(i + B, len(prompts))}/{len(prompts)} ({time.time() - t0:.0f}s)')
    except Exception as e:
        print(f'FAILED at batch starting index {i} (rows {i}-{i+B}): {type(e).__name__}: {e}')
        import traceback
        traceback.print_exc()
        raise
    finally:
        torch.cuda.empty_cache()
tokenizer.padding_side = old_side

records = []
for r, p in zip(val_rows, preds):
    res = eps_declared(r, p)
    records.append({
        'id': r.get('id'),
        'bucket': r.get('bucket'),
        'difficulty': r.get('difficulty'),
        'category': r.get('category'),
        'exact_match': float(p.strip() == r['output'].strip()),
        'ned': round(ned(p, r['output']), 4),
        'wer': round(jiwer.wer(r['output'], p), 4) if p.strip() else 1.0,
        'eps': None if res is None else round(res[0], 4),
        'missing_entities': [] if res is None else res[1],
        'pred': p,
        'ref': r['output'],
    })

def agg(rows):
    rows = list(rows)
    if not rows:
        return {}
    eps_rows = [x for x in rows if x['eps'] is not None]
    return {
        'n': len(rows),
        'exact_match': round(sum(x['exact_match'] for x in rows) / len(rows), 4),
        'ned': round(sum(x['ned'] for x in rows) / len(rows), 4),
        'wer': round(sum(x['wer'] for x in rows) / len(rows), 4),
        'eps_mean': round(sum(x['eps'] for x in eps_rows) / len(eps_rows), 4) if eps_rows else None,
        'eps_hard_fail': round(sum(1 for x in eps_rows if x['eps'] < 1.0) / len(eps_rows), 4) if eps_rows else None,
    }

def group_by(key):
    groups = collections.defaultdict(list)
    for x in records:
        groups[x.get(key)].append(x)
    return {str(k): agg(v) for k, v in sorted(groups.items(), key=lambda kv: str(kv[0]))}

report = {
    'model': 'merged_fp16',
    'eval_n': len(records),
    'overall': agg(records),
    'by_bucket': group_by('bucket'),
    'by_difficulty': group_by('difficulty'),
}

print(json.dumps(report['overall'], indent=1))
EVAL_OUT.write_text(json.dumps({'report': report, 'records': records}, ensure_ascii=False), encoding='utf-8')
print('Evaluation complete and report saved to:', EVAL_OUT)

Loading merged model for evaluation...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Evaluating 300 validation rows...
  4/300 (4s)
  8/300 (6s)
  12/300 (8s)
  16/300 (12s)
  20/300 (13s)
  24/300 (15s)
  28/300 (17s)
  32/300 (19s)
  36/300 (20s)
  40/300 (22s)
  44/300 (25s)
  48/300 (27s)
  52/300 (29s)
  56/300 (31s)
  60/300 (32s)
  64/300 (34s)
  68/300 (37s)
  72/300 (40s)
  76/300 (41s)
  80/300 (44s)
  84/300 (46s)
  88/300 (48s)
  92/300 (51s)
  96/300 (53s)
  100/300 (55s)
  104/300 (57s)
  108/300 (59s)
  112/300 (62s)
  116/300 (64s)
  120/300 (66s)
  124/300 (68s)
  128/300 (70s)
  132/300 (72s)
  136/300 (73s)
  140/300 (75s)
  144/300 (78s)
  148/300 (80s)
  152/300 (81s)
  156/300 (84s)
  160/300 (85s)
  164/300 (87s)
  168/300 (89s)
  172/300 (91s)
  176/300 (93s)
  180/300 (95s)
  184/300 (97s)
  188/300 (99s)
  192/300 (101s)
  196/300 (104s)
  200/300 (106s)
  204/300 (108s)
  208/300 (111s)
  212/300 (112s)
  216/300 (114s)
  220/300 (116s)
  224/300 (118s)
  228/300 (120s)
  232/300 (121s)
  236/300 (123s)
  240/300 (125s)
  244/300 (127s)
  248